# Глава 10. CatBoost и LightGBM на своём компьютере

Практика к урокам «CatBoost и LightGBM в своём Jupyter» и «Проект: прогноз дневной выручки». В браузере курса этих библиотек нет: это скомпилированный код на C++, его не собирают для Python в браузере. Поэтому работаем в своём Jupyter.

**Что сделаем**

1. Возьмём данные и признаки проекта главы 10: `daily_sales.csv`, выручка шести кофеен по дням.
2. Разделим данные по времени на три части: обучение, валидация для ранней остановки и тест.
3. Обучим бустинги с ранней остановкой: `HistGradientBoostingRegressor` из sklearn с флагами `get_dummies` и с нативными категориями, LightGBM в двух настройках и CatBoost.
4. Соберём таблицу «модель | RMSE | MAE | смещение | деревьев | fit, с | predict, мс».

**Перед запуском**

- Установите библиотеки в окружение курса (глава 2): `uv add catboost lightgbm` или `pip install catboost lightgbm`.
- Блокнот сам ищет `daily_sales.csv`: в `../../data` (папка курса), `../data`, `data` или рядом с собой.
- Все библиотеки работают на одном потоке процессора (`THREADS = 1` в первой ячейке). Так сравнение времени честнее: каждая библиотека по-своему использует ядра, и на загруженном компьютере многопоточность может даже замедлить обучение. Хотите быстрее — поставьте `THREADS` равным числу ядер и перезапустите блокнот.
- Сохранённые выводы получены на нашем компьютере. Секунды у вас будут другими, а качество (RMSE, MAE) должно совпасть с точностью до версий библиотек.

In [1]:
import os
THREADS = 1                                     # все библиотеки на одном потоке, см. пояснение выше
os.environ["OMP_NUM_THREADS"] = str(THREADS)    # задаём до импорта sklearn и lightgbm

import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import catboost
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 0
DATA = next(p for p in [Path("../../data"), Path("../data"), Path("data"), Path(".")]
            if (p / "daily_sales.csv").exists())

print("pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lgb.__version__, "| catboost", catboost.__version__)
print("данные:", DATA.resolve())

pandas 3.0.5 | sklearn 1.9.1 | lightgbm 4.7.0 | catboost 1.2.10
данные: …/ds-course/site/data


## 1. Данные и признаки

Признаки те же, что в функции `prep` из проекта: температура, дождь, праздник, акция, день недели, месяц, выходной, номер дня `day_n`, кофейня и формат. Разница одна: кофейню и формат оставляем **текстом**, без `get_dummies`. Каждая библиотека получит категории в том виде, который понимает сама.

In [2]:
s = pd.read_csv(DATA / "daily_sales.csv", parse_dates=["date"])
s = s.sort_values(["shop_id", "date"]).reset_index(drop=True)

def prep(s):
    s = s.copy()
    s["weekday"] = s["date"].dt.weekday
    s["month"] = s["date"].dt.month
    s["weekend"] = (s["weekday"] >= 5).astype(int)
    s["day_n"] = (s["date"] - s["date"].min()).dt.days
    cols = ["temp", "rain", "holiday", "promo", "weekday", "month", "weekend", "day_n", "shop_id", "format"]
    return s[cols]

CAT = ["shop_id", "format"]
X = prep(s)                                                     # категории — строки (для CatBoost)
X_cat = X.astype({c: "category" for c in CAT})                  # тип category (для LightGBM и HGB)
X_dum = pd.get_dummies(X, columns=CAT, dtype=int)               # флаги (классический вариант)
y = s["revenue_k"]

print("строк:", len(X), "| признаков:", X.shape[1], "| столбцов после get_dummies:", X_dum.shape[1])
X.head()

строк: 4386 | признаков: 10 | столбцов после get_dummies: 17


,temp,rain,holiday,promo,weekday,month,weekend,day_n,shop_id,format
0,-19.0,0,1,0,0,1,0,0,KZN-1,улица
1,-14.4,1,1,0,1,1,0,1,KZN-1,улица
2,-10.8,0,1,0,2,1,0,2,KZN-1,улица
3,-16.8,1,1,0,3,1,0,3,KZN-1,улица
4,-18.7,0,1,0,4,1,0,4,KZN-1,улица


**Вывод по разделу 1.** Один набор признаков в трёх видах: строки, тип `category` и флаги. После `get_dummies` столбцов становится 17 вместо 10. На шести кофейнях это не страшно, но с тысячами товаров или адресов таблица флагов раздулась бы до тысяч почти пустых столбцов.

## 2. Разбиение по времени

Как в проекте: прогноз нужен на будущее, поэтому проверяем «прошлое → будущее».

- **обучение** (`fit`) — до 1 апреля 2025;
- **валидация** (`val`) — апрель–июнь 2025: по ней ранняя остановка выбирает число деревьев;
- **тест** (`test`) — июль–декабрь 2025: его модели не видят до финальной оценки.

Тест для ранней остановки брать нельзя. Иначе число деревьев подобрано по тесту, и оценка на нём получится оптимистичной.

In [3]:
fit = s["date"] < "2025-04-01"
val = (s["date"] >= "2025-04-01") & (s["date"] < "2025-07-01")
test = s["date"] >= "2025-07-01"

for name, mask in [("fit", fit), ("val", val), ("test", test)]:
    print(f"{name:5} {mask.sum():5} строк, {s.loc[mask, 'date'].min():%Y-%m-%d} — {s.loc[mask, 'date'].max():%Y-%m-%d}")

fit    2736 строк, 2024-01-01 — 2025-03-31
val     546 строк, 2025-04-01 — 2025-06-30
test   1104 строк, 2025-07-01 — 2025-12-31


**Вывод по разделу 2.** Три непересекающихся периода идут друг за другом. Все модели учатся на одном и том же `fit`, останавливаются по одному `val` и проверяются на одном `test`, поэтому сравнение честное.

## 3. Обучение и замер

Для каждой модели запишем строку таблицы: качество на тесте (RMSE и MAE, тыс. ₽), смещение (средняя ошибка «факт − прогноз»: больше нуля — модель занижает), сколько деревьев выбрала ранняя остановка, время обучения и время прогноза. Секундомер — `time.perf_counter()`, как в главе 9. Прогноз быстрый, поэтому его повторяем 5 раз и берём лучшее время.

In [4]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def timed(f, repeat=1):
    # Запускает f() repeat раз; возвращает результат и лучшее время в секундах.
    best = None
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = f()
        t = time.perf_counter() - t0
        best = t if best is None else min(best, t)
    return result, best

rows = []

def add_row(name, model, X_test, n_trees, fit_s):
    pred, pred_s = timed(lambda: model.predict(X_test), repeat=5)
    rows.append({"модель": name, "RMSE": rmse(y[test], pred), "MAE": mean_absolute_error(y[test], pred),
                 "смещение": float((y[test] - pred).mean()),        # > 0: прогноз в среднем занижен
                 "деревьев": int(n_trees), "fit, с": fit_s, "predict, мс": pred_s * 1000})
    print(rows[-1])

### HistGradientBoosting из sklearn

Встроенная ранняя остановка HGB берёт валидацию случайной долей строк, а нам нужна валидация по времени. Поэтому делаем, как в шаге 6 проекта: обучаем 1000 деревьев, по `staged_predict` находим лучшее число деревьев на `val` и переобучаем модель с этим числом. В таблицу идёт время последнего обучения.

In [5]:
def hgb_by_val(X_all, **extra):
    params = dict(learning_rate=0.05, max_iter=1000, early_stopping=False, random_state=RANDOM_STATE, **extra)
    probe = HistGradientBoostingRegressor(**params).fit(X_all[fit], y[fit])
    curve = [rmse(y[val], p) for p in probe.staged_predict(X_all[val])]
    params["max_iter"] = int(np.argmin(curve)) + 1
    model, fit_s = timed(lambda: HistGradientBoostingRegressor(**params).fit(X_all[fit], y[fit]))
    return model, params["max_iter"], fit_s

m, k, t = hgb_by_val(X_dum)
add_row("HGB, get_dummies", m, X_dum[test], k, t)
m, k, t = hgb_by_val(X_cat, categorical_features="from_dtype")
add_row("HGB, категории", m, X_cat[test], k, t)

{'модель': 'HGB, get_dummies', 'RMSE': 7.346061824864619, 'MAE': 5.30354378011953, 'смещение': 1.7513823643084254, 'деревьев': 163, 'fit, с': 0.3527454900013254, 'predict, мс': 8.800682000583038}


{'модель': 'HGB, категории', 'RMSE': 7.274325553241675, 'MAE': 5.266562817273766, 'смещение': 2.0998253712299944, 'деревьев': 191, 'fit, с': 0.4874801089990797, 'predict, мс': 15.571707999697537}


### LightGBM

Категории LightGBM понимает по типу `category`. Валидацию передают в `eval_X` и `eval_y` (так с версии 4.7; в старых версиях — `eval_set=[(X_val, y_val)]`). Ранняя остановка — обратный вызов `lgb.early_stopping(100)`: если 100 деревьев подряд не улучшили ошибку на валидации, обучение останавливается. Лучшее число деревьев хранится в `best_iteration_`, и `predict` использует именно его.

Обучим две версии. Первая — с настройками по умолчанию: до 31 листа в дереве и не меньше 20 строк в листе. Эти настройки рассчитаны на большие таблицы. Вторая — с мелкими деревьями: 7 листьев и 10 строк в листе.

In [6]:
for name, leaves, min_rows in [("LightGBM", 31, 20), ("LightGBM, 7 листьев", 7, 10)]:
    lg = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=leaves, min_child_samples=min_rows,
                           random_state=RANDOM_STATE, verbose=-1, n_jobs=THREADS)
    _, fit_s = timed(lambda: lg.fit(X_cat[fit], y[fit], eval_X=X_cat[val], eval_y=y[val],
                                    callbacks=[lgb.early_stopping(100, verbose=False)]))
    add_row(name, lg, X_cat[test], lg.best_iteration_, fit_s)

{'модель': 'LightGBM', 'RMSE': 7.88020740329592, 'MAE': 5.642047983369115, 'смещение': 3.4882166766582166, 'деревьев': 137, 'fit, с': 0.12845709400062333, 'predict, мс': 7.652038999367505}


{'модель': 'LightGBM, 7 листьев', 'RMSE': 6.984512007560178, 'MAE': 5.210835593188687, 'смещение': 2.0810878123034464, 'деревьев': 702, 'fit, с': 0.22958159700101533, 'predict, мс': 32.356899999285815}


### CatBoost

CatBoost получает категории строками и список их имён в `cat_features`. Внутри он заменяет категорию средним ответом по ней, посчитанным только по «предыдущим» строкам: так ответ текущей строки не подсказывает сам себе. `early_stopping_rounds=100` и `use_best_model=True` — та же ранняя остановка по `eval_set`. `allow_writing_files=False` — не создавать рядом с блокнотом папку `catboost_info` с логами обучения.

In [7]:
cb = CatBoostRegressor(iterations=3000, learning_rate=0.05, depth=6, random_seed=RANDOM_STATE,
                       verbose=0, early_stopping_rounds=100, thread_count=THREADS, allow_writing_files=False)
_, fit_s = timed(lambda: cb.fit(X[fit], y[fit], cat_features=CAT,
                                eval_set=(X[val], y[val]), use_best_model=True))
add_row("CatBoost", cb, X[test], cb.get_best_iteration() + 1, fit_s)

{'модель': 'CatBoost', 'RMSE': 6.551519847544615, 'MAE': 4.933918772835, 'смещение': 0.11852843090721614, 'деревьев': 706, 'fit, с': 3.6447292679986276, 'predict, мс': 3.588051000406267}


**Вывод по разделу 3.** Все пять моделей обучены на одних данных и остановлены по одной валидации. Число деревьев у них разное: у каждой библиотеки свой размер и своё устройство дерева. Поэтому сравнивать модели надо по качеству на тесте и по времени, а не по числу деревьев.

## 4. Сравнение

In [8]:
table = pd.DataFrame(rows).set_index("модель").sort_values("RMSE")
table.round({"RMSE": 2, "MAE": 2, "смещение": 2, "fit, с": 2, "predict, мс": 1})

,RMSE,MAE,смещение,деревьев,"fit, с","predict, мс"
модель,,,,,,
CatBoost,6.55,4.93,0.12,706,3.64,3.6
"LightGBM, 7 листьев",6.98,5.21,2.08,702,0.23,32.4
"HGB, категории",7.27,5.27,2.10,191,0.49,15.6
"HGB, get_dummies",7.35,5.30,1.75,163,0.35,8.8
LightGBM,7.88,5.64,3.49,137,0.13,7.7


In [9]:
imp = pd.Series(cb.get_feature_importance(), index=X.columns).sort_values(ascending=False)
imp.round(1)

format     27.3
weekday    19.9
weekend    16.0
shop_id    11.9
rain        6.4
holiday     6.0
promo       5.8
day_n       2.9
temp        2.5
month       1.4
dtype: float64

**Вывод по разделу 4.**

- **Качество.** Лучший на тесте — CatBoost: средняя ошибка 4.93 тыс. ₽ в день (RMSE 6.55) против 5.27–5.30 у HGB. LightGBM с настройками по умолчанию — худший (5.64): 31 лист и не меньше 20 строк в листе рассчитаны на большие таблицы, а у нас в обучении 2736 строк. С мелкими деревьями (7 листьев) LightGBM поднимается на второе место (5.21). На маленьких данных сравнение «из коробки» — во многом сравнение настроек по умолчанию.
- **Масштаб разницы.** Между лучшей и худшей моделью около 0.7 тыс. ₽ средней ошибки при средней выручке во втором полугодии около 66 тыс. ₽ в день. В проекте бустинг выиграл у линейной модели больше 7 тыс. ₽. Признаки и честная проверка важнее выбора библиотеки.
- **Смещение.** HGB и LightGBM в среднем занижают прогноз на 1.75–3.49 тыс. ₽: сеть растёт, а деревья не продолжают тренд. CatBoost на этом периоде почти не смещён (0.12), но рассчитывать на это нельзя: тренд надёжнее снимать явно, как в шаге 7 проекта.
- **Время.** На четырёх тысячах строк все модели обучаются за секунды. LightGBM обучился быстрее всех, CatBoost — дольше всех: около 700 деревьев глубины 6. Зато прогноз у CatBoost самый быстрый: его деревья симметричные, на каждом уровне одно и то же условие. Секунды шумят: при повторных запусках они менялись в полтора-два раза, а соседние по времени модели иногда менялись местами. Устойчивыми оставались только крайние места: быстрее всех обучается LightGBM, дольше всех — CatBoost, прогноз самый быстрый у CatBoost. Время начинает решать на миллионах строк и при частом переобучении.
- **Категории.** HGB с нативными категориями чуть точнее флагов (MAE 5.27 против 5.30) и обходится без раздувания таблицы. Самые важные признаки для CatBoost — формат кофейни, день недели и выходной: из них в шаге 1 проекта и сложилось взаимодействие «формат × выходной», которое деревья находят сами.

## 5. Попробуйте сами

1. Поменяйте `learning_rate` на 0.1 и 0.02 у LightGBM и CatBoost. Как меняются число деревьев, время и RMSE?
2. Передайте в CatBoost таблицу `X_dum` без `cat_features`. Хуже ли стало качество?
3. Перенесите сюда приём из шага 7 проекта: обучите бустинги на `y / trend` без `day_n` и сравните RMSE.
4. Попробуйте у LightGBM `num_leaves=15` и `min_child_samples=5`. У LightGBM дерево растёт «по листьям», и его сложность задаёт `num_leaves`, а не глубина.